# Forecasting Baselines and Evaluation From Scratch

## 1. Why Baselines Come Before Deep Learning

A deep forecasting model should not be considered useful simply because it produces predictions.

It should outperform meaningful reference methods.

A forecasting baseline answers:

> What performance can we achieve with a very simple assumption?

If a later LSTM, Transformer, or foundation model cannot beat a naive forecast, the added complexity may not be justified.

![Forecasting evaluation logic](images/05_forecasting_evaluation.png)

## 2. Learning Objectives

By the end of this notebook, you should be able to:

- construct naive, seasonal-naive, and moving-average forecasts,
- calculate MAE, MSE, RMSE, MAPE, sMAPE, and MASE,
- understand why percentage metrics can fail near zero,
- compare models over the same test period,
- inspect residuals and residual autocorrelation,
- understand walk-forward and rolling-origin evaluation,
- evaluate error by forecast horizon,
- construct a simple empirical prediction interval,
- distinguish point forecasts from uncertainty estimates.

## 3. Load and Clean the Time Series

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data_path = "data/synthetic_hourly_demand.csv"

data = pd.read_csv(data_path)

data["timestamp"] = pd.to_datetime(data["timestamp"])

data = data.set_index("timestamp")

data["demand"] = data["demand"].interpolate(method="time")

series = data["demand"]

print("Series length:")
print(len(series))

print()

print(series.head())

## 4. Create a Chronological Train/Test Split

For this notebook:

```text
80% training
20% testing
```

The test period is strictly later than the training period.

In [ ]:
split_index = int(len(series) * 0.80)

train_series = series.iloc[:split_index]

test_series = series.iloc[split_index:]

print("Training observations:")
print(len(train_series))

print()

print("Test observations:")
print(len(test_series))

## 5. Baseline 1: Naive Forecast

The naive one-step-ahead rule is:

\[
\hat{y}_t = y_{t-1}
\]

The next value is assumed to equal the most recently observed value.

In [ ]:
naive_predictions = []

previous_value = train_series.iloc[-1]

for actual_value in test_series:
    prediction = previous_value

    naive_predictions.append(prediction)

    previous_value = actual_value

naive_predictions = np.array(naive_predictions)

print("Naive predictions:")
print(len(naive_predictions))

Because the actual observation is added after each step, this is a **one-step-ahead walk-forward** evaluation.

## 6. Baseline 2: Seasonal Naive

For hourly data with daily seasonality:

\[
\hat{y}_t = y_{t-24}
\]

The model assumes that the current hour resembles the same hour one day earlier.

In [ ]:
season_length = 24

history = list(train_series.to_numpy())

seasonal_naive_predictions = []

for actual_value in test_series:
    prediction = history[-season_length]

    seasonal_naive_predictions.append(prediction)

    history.append(actual_value)

seasonal_naive_predictions = np.array(seasonal_naive_predictions)

print("Seasonal naive predictions:")
print(len(seasonal_naive_predictions))

## 7. Baseline 3: Moving Average

A moving-average forecast uses recent history:

\[
\hat{y}_t =
rac{1}{w}
\sum_{i=1}^{w}
y_{t-i}
\]

In [ ]:
moving_average_window = 24

history = list(train_series.to_numpy())

moving_average_predictions = []

for actual_value in test_series:
    recent_values = history[-moving_average_window:]

    prediction = np.mean(recent_values)

    moving_average_predictions.append(prediction)

    history.append(actual_value)

moving_average_predictions = np.array(moving_average_predictions)

print("Moving-average predictions:")
print(len(moving_average_predictions))

## 8. Visualize the Baseline Forecasts

In [ ]:
plot_length = 24 * 7

actual_plot = test_series.iloc[:plot_length]

plt.figure(figsize=(14, 5))

plt.plot(actual_plot.index, actual_plot.values, label="Actual")

plt.plot(
    actual_plot.index,
    naive_predictions[:plot_length],
    label="Naive"
)

plt.plot(
    actual_plot.index,
    seasonal_naive_predictions[:plot_length],
    label="Seasonal Naive"
)

plt.plot(
    actual_plot.index,
    moving_average_predictions[:plot_length],
    label="Moving Average"
)

plt.xlabel("Time")
plt.ylabel("Demand")
plt.title("Baseline Forecasts for the First Test Week")
plt.legend()

plt.show()

## 9. Forecast Error

\[
e_t = y_t-\hat{y}_t
\]

Positive error means the actual value was above the prediction.

Negative error means the prediction was above the actual value.

In [ ]:
actual_values = test_series.to_numpy()

naive_errors = actual_values - naive_predictions

print("First ten naive errors:")
print(naive_errors[:10])

## 10. Mean Absolute Error (MAE)

\[
MAE =
rac{1}{n}
\sum |y_i-\hat{y}_i|
\]

MAE is measured in the same unit as the target.

In [ ]:
def calculate_mae(actual, predicted):
    absolute_errors = np.abs(actual - predicted)

    mae = np.mean(absolute_errors)

    return mae

## 11. Mean Squared Error (MSE)

\[
MSE =
rac{1}{n}
\sum (y_i-\hat{y}_i)^2
\]

Large errors receive a stronger penalty.

In [ ]:
def calculate_mse(actual, predicted):
    squared_errors = (actual - predicted) ** 2

    mse = np.mean(squared_errors)

    return mse

## 12. Root Mean Squared Error (RMSE)

\[
RMSE = \sqrt{MSE}
\]

RMSE returns the squared-error metric to the original target unit.

In [ ]:
def calculate_rmse(actual, predicted):
    mse = calculate_mse(actual, predicted)

    rmse = np.sqrt(mse)

    return rmse

## 13. Mean Absolute Percentage Error (MAPE)

\[
MAPE =
rac{100}{n}
\sum
\left|
rac{y_i-\hat{y}_i}{y_i}
ight|
\]

MAPE is intuitive as a percentage, but it becomes unstable when actual values approach zero.

In [ ]:
def calculate_mape(actual, predicted):
    actual = np.array(actual)

    predicted = np.array(predicted)

    nonzero_mask = actual != 0

    percentage_errors = np.abs(
        (actual[nonzero_mask] - predicted[nonzero_mask])
        / actual[nonzero_mask]
    )

    mape = np.mean(percentage_errors) * 100.0

    return mape

## 14. Symmetric MAPE (sMAPE)

One common form is:

\[
sMAPE =
rac{100}{n}
\sum
rac{2|y_i-\hat{y}_i|}{|y_i|+|\hat{y}_i|}
\]

In [ ]:
def calculate_smape(actual, predicted):
    actual = np.array(actual)

    predicted = np.array(predicted)

    numerator = 2.0 * np.abs(actual - predicted)

    denominator = np.abs(actual) + np.abs(predicted)

    valid_mask = denominator != 0

    smape = np.mean(
        numerator[valid_mask] / denominator[valid_mask]
    ) * 100.0

    return smape

## 15. Mean Absolute Scaled Error (MASE)

MASE compares model error with a naive in-sample error scale.

\[
MASE =
rac{MAE_{model}}{MAE_{naive\ scale}}
\]

A value below 1 indicates performance better than the corresponding naive scale.

In [ ]:
def calculate_mase(training_values, actual, predicted):
    training_values = np.array(training_values)

    naive_training_errors = np.abs(
        training_values[1:] - training_values[:-1]
    )

    scale = np.mean(naive_training_errors)

    model_mae = calculate_mae(actual, predicted)

    mase = model_mae / scale

    return mase

## 16. Evaluate All Baselines

In [ ]:
baseline_names = [
    "Naive",
    "Seasonal Naive",
    "Moving Average"
]

baseline_predictions = [
    naive_predictions,
    seasonal_naive_predictions,
    moving_average_predictions
]

results = []

for name, predictions in zip(baseline_names, baseline_predictions):
    mae = calculate_mae(actual_values, predictions)

    rmse = calculate_rmse(actual_values, predictions)

    mape = calculate_mape(actual_values, predictions)

    smape = calculate_smape(actual_values, predictions)

    mase = calculate_mase(
        train_series.to_numpy(),
        actual_values,
        predictions
    )

    results.append(
        {
            "Model": name,
            "MAE": mae,
            "RMSE": rmse,
            "MAPE": mape,
            "sMAPE": smape,
            "MASE": mase
        }
    )

results_table = pd.DataFrame(results)

print(results_table)

## 17. Rank the Baselines

In [ ]:
ranked_results = results_table.sort_values(by="MAE")

print(ranked_results)

## 18. Why One Metric Is Not Enough

Different metrics answer different questions.

### MAE
Typical absolute miss.

### RMSE
More sensitive to large errors.

### MAPE
Percentage interpretation but problematic near zero.

### MASE
Scales error relative to a naive benchmark.

## 19. Residual Analysis

Residuals are:

\[
e_t = y_t-\hat{y}_t
\]

A useful forecasting method should not leave strong predictable patterns in the residuals.

In [ ]:
seasonal_residuals = actual_values - seasonal_naive_predictions

print("Residual mean:")
print(seasonal_residuals.mean())

print()

print("Residual standard deviation:")
print(seasonal_residuals.std())

In [ ]:
plt.figure(figsize=(14, 4))

plt.plot(test_series.index, seasonal_residuals)

plt.axhline(0.0)

plt.xlabel("Time")
plt.ylabel("Residual")
plt.title("Seasonal-Naive Residuals")

plt.show()

## 20. Residual Autocorrelation

If residuals remain strongly autocorrelated, the baseline has left temporal structure unexplained.

In [ ]:
residual_series = pd.Series(seasonal_residuals)

residual_lags = [1, 2, 6, 12, 24, 48]

for lag in residual_lags:
    correlation = residual_series.autocorr(lag=lag)

    print(
        "Lag:",
        lag,
        "| Residual autocorrelation:",
        round(correlation, 4)
    )

## 21. Walk-Forward Evaluation

Conceptually:

```text
Train → Forecast next
Train + observed value → Forecast next
Train + another observed value → Forecast next
...
```

This mimics repeated forecasting in operation.

## 22. Rolling-Origin Evaluation

A more general setup evaluates several forecast origins and several future horizons.

```text
Origin 1 → t+1, t+2, t+3
Origin 2 → t+1, t+2, t+3
Origin 3 → t+1, t+2, t+3
```

## 23. Multi-Horizon Seasonal Baseline

In [ ]:
full_values = series.to_numpy()

maximum_horizon = 6

starting_origin = split_index

ending_origin = len(full_values) - maximum_horizon

horizon_errors = {}

for horizon in range(1, maximum_horizon + 1):
    horizon_errors[horizon] = []

for origin in range(starting_origin, ending_origin):
    for horizon in range(1, maximum_horizon + 1):
        target_index = origin + horizon - 1

        seasonal_index = target_index - season_length

        prediction = full_values[seasonal_index]

        actual = full_values[target_index]

        absolute_error = np.abs(actual - prediction)

        horizon_errors[horizon].append(absolute_error)

In [ ]:
horizon_mae = []

for horizon in range(1, maximum_horizon + 1):
    mae = np.mean(horizon_errors[horizon])

    horizon_mae.append(mae)

    print(
        "Horizon:",
        horizon,
        "| MAE:",
        round(mae, 4)
    )

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(
    range(1, maximum_horizon + 1),
    horizon_mae,
    marker="o"
)

plt.xlabel("Forecast Horizon")
plt.ylabel("MAE")
plt.title("Error by Forecast Horizon")

plt.show()

## 24. Why Horizon-Wise Evaluation Matters

A model may be strong at:

```text
t + 1
```

and weak at:

```text
t + 24
```

One aggregate score can hide this degradation.

## 25. Point Forecast vs Prediction Interval

Point forecast:

```text
Demand = 58.4
```

Prediction interval:

```text
Forecast = 58.4
Approx. 80% interval = [53.1, 63.7]
```

Intervals communicate uncertainty.

## 26. Simple Empirical Prediction Interval

For learning purposes, use residual quantiles from the training period.

This is not a complete probabilistic forecasting model, but it demonstrates the basic idea.

In [ ]:
training_values = train_series.to_numpy()

training_seasonal_errors = (
    training_values[season_length:]
    - training_values[:-season_length]
)

lower_quantile = np.quantile(
    training_seasonal_errors,
    0.10
)

upper_quantile = np.quantile(
    training_seasonal_errors,
    0.90
)

print("10th residual quantile:")
print(lower_quantile)

print()

print("90th residual quantile:")
print(upper_quantile)

In [ ]:
lower_bound = seasonal_naive_predictions + lower_quantile

upper_bound = seasonal_naive_predictions + upper_quantile

plot_length = 24 * 5

plt.figure(figsize=(14, 5))

plt.plot(
    test_series.index[:plot_length],
    actual_values[:plot_length],
    label="Actual"
)

plt.plot(
    test_series.index[:plot_length],
    seasonal_naive_predictions[:plot_length],
    label="Seasonal Naive"
)

plt.fill_between(
    test_series.index[:plot_length],
    lower_bound[:plot_length],
    upper_bound[:plot_length],
    alpha=0.25,
    label="Approx. 80% Interval"
)

plt.xlabel("Time")
plt.ylabel("Demand")
plt.title("Point Forecast and Empirical Prediction Interval")
plt.legend()

plt.show()

## 27. Interval Coverage

Coverage estimates the fraction of actual observations inside the interval.

In [ ]:
inside_interval = (
    (actual_values >= lower_bound)
    & (actual_values <= upper_bound)
)

coverage = np.mean(inside_interval)

print("Empirical interval coverage:")
print(coverage)

## 28. Why Uncertainty Is Harder Than Point Prediction

A good uncertainty model must consider:

- changing variance,
- forecast horizon,
- temporal dependence,
- distribution shift,
- model uncertainty,
- data noise.

Later advanced methods may use quantile loss, probabilistic forecasting, or conformal prediction.

## 29. Fair Model Comparison

Every model should use the same:

```text
training period
validation period
test period
target
lookback rules
forecast horizon
metrics
```

Otherwise, metric differences may reflect experimental design rather than model quality.

## 30. Benchmark Table for the Next Folder

| Model | MAE | RMSE | MAPE | sMAPE | MASE |
|---|---:|---:|---:|---:|---:|
| Naive | ✓ | ✓ | ✓ | ✓ | ✓ |
| Seasonal Naive | ✓ | ✓ | ✓ | ✓ | ✓ |
| Moving Average | ✓ | ✓ | ✓ | ✓ | ✓ |
| Dense Network | next folder | | | | |
| 1D CNN | next folder | | | | |
| RNN | next folder | | | | |
| LSTM | next folder | | | | |
| GRU | next folder | | | | |
| TCN | next folder | | | | |
| Transformer | next folder | | | | |
| Time-Series Foundation Model | next folder | | | | |

These baselines become the reference point for more complex forecasting architectures.

## 31. Deeper Evaluation Concepts

When two models have very similar average errors, one number may not prove that one method is meaningfully better.

More advanced comparison can examine:

- paired forecast errors,
- confidence intervals,
- error distributions,
- Diebold–Mariano-type tests.

This notebook introduces the issue so later portfolio claims remain appropriately cautious.

## 32. Connection to Deep Learning for Time Series Forecasting

We now have:

```text
valid split
    ↓
valid windows
    ↓
simple baselines
    ↓
forecast metrics
    ↓
rolling evaluation
    ↓
uncertainty concept
```

The next folder can now ask whether Dense networks, CNNs, RNNs, LSTMs, GRUs, TCNs, Transformers, and Time-Series Foundation Models improve on these benchmarks.

## 33. Key Takeaways

- Forecasting models should beat meaningful simple baselines.
- Naive, seasonal-naive, and moving-average methods provide interpretable references.
- MAE, RMSE, MAPE, sMAPE, and MASE emphasize different error behavior.
- Residual analysis can reveal unexplained temporal structure.
- Walk-forward and rolling-origin evaluation mimic repeated future prediction.
- Multi-step forecasts should be evaluated by horizon.
- Prediction intervals communicate uncertainty that point forecasts cannot.
- Fair comparison requires identical temporal splits and forecast definitions.